# SVO Grammar Coach (Step 3) - GUI Prototype

We build a simple GUI in the notebook with `ipywidgets`.
The interface guides the user through subject, verb, and object choices.

Note:
- If you do not have `ipywidgets`, install it with `pip install ipywidgets`.
- This GUI is intentionally minimal and easy to navigate.


In [1]:
# GUI dependencies
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as exc:
    raise ImportError("ipywidgets is required. Run: pip install ipywidgets") from exc


In [2]:
import json
from pathlib import Path

paths = [
    Path("notebooks_exc_2/svo_resources.json"),
    Path("svo_resources.json"),
]

resources = None
for p in paths:
    if p.exists():
        resources = json.loads(p.read_text())
        break

if resources is None:
    raise FileNotFoundError("svo_resources.json not found. Run Step 1 first.")

NOUNS = resources["nouns"]
ADJECTIVES = resources["adjectives"]
VERBS = resources["verbs"]


In [3]:
VOWELS = set("aeiou")

POSSESSIVE = {
    "my": "my",
    "your": "your",
    "his": "his",
    "her": "her",
    "its": "its",
    "our": "our",
    "their": "their",
}


def plural_form(noun):
    return NOUNS[noun]["plural"]


def choose_indefinite(next_word):
    if not next_word:
        return "a"
    return "an" if next_word[0].lower() in VOWELS else "a"


def determiner(d_type, number, next_word=None, possessor_key=None):
    if d_type == "none":
        return ""
    if d_type == "definite":
        return "the"
    if d_type == "indefinite":
        if number == "pl":
            return ""
        return choose_indefinite(next_word)
    if d_type == "possessive":
        return POSSESSIVE.get(possessor_key or "my", "my")
    if d_type == "demonstrative":
        return "this" if number == "sg" else "these"
    raise ValueError("Unknown determiner type")


def make_noun_phrase(
    noun,
    number="sg",
    d_type="definite",
    adjective=None,
    possessor_key=None,
):
    head = noun if number == "sg" else plural_form(noun)
    next_word = adjective or head
    det = determiner(d_type, number, next_word=next_word, possessor_key=possessor_key)

    parts = []
    if det:
        parts.append(det)
    if adjective:
        parts.append(adjective)
    parts.append(head)
    return " ".join(parts)


def present_3sg(verb):
    if verb.endswith("y") and len(verb) >= 2 and verb[-2] not in VOWELS:
        return verb[:-1] + "ies"
    if verb.endswith(("s", "sh", "ch", "x", "z", "o")):
        return verb + "es"
    return verb + "s"


def past_regular(verb):
    if verb.endswith("e"):
        return verb + "d"
    if verb.endswith("y") and len(verb) >= 2 and verb[-2] not in VOWELS:
        return verb[:-1] + "ied"
    return verb + "ed"


def conjugate_present(verb, person, number):
    if person == 3 and number == "sg":
        return present_3sg(verb)
    return verb


def do_aux(tense, person, number):
    if tense == "past":
        return "did"
    if person == 3 and number == "sg":
        return "does"
    return "do"


def imperative_form(verb):
    return verb


def would_form():
    return "would"

PRONOUNS = {
    (1, "sg"): "I",
    (2, "sg"): "you",
    (3, "sg", "m"): "he",
    (3, "sg", "f"): "she",
    (3, "sg", "n"): "it",
    (1, "pl"): "we",
    (2, "pl"): "you",
    (3, "pl", "pl"): "they",
}


def sentence_case(s):
    if not s:
        return s
    return s[0].upper() + s[1:]


def build_sentence(
    subject_phrase,
    verb_inf,
    object_phrase,
    sentence_type="affirmative",
    tense="present",
    person=3,
    number="sg",
):
    if sentence_type == "subjunctive":
        core = f"{subject_phrase} {would_form()} {verb_inf} {object_phrase}".strip()
        return sentence_case(core) + "."

    if sentence_type == "imperative":
        core = f"{imperative_form(verb_inf)} {object_phrase}".strip()
        return sentence_case(core) + "!"

    if sentence_type == "question":
        aux = do_aux(tense, person, number)
        core = f"{aux} {subject_phrase} {verb_inf} {object_phrase}".strip()
        return sentence_case(core) + "?"

    if sentence_type == "negative":
        aux = do_aux(tense, person, number)
        core = f"{subject_phrase} {aux} not {verb_inf} {object_phrase}".strip()
        return sentence_case(core) + "."

    if tense == "past":
        verb_form = past_regular(verb_inf)
    else:
        verb_form = conjugate_present(verb_inf, person, number)

    core = f"{subject_phrase} {verb_form} {object_phrase}".strip()
    return sentence_case(core) + "."


In [4]:
# --- Widgets ---

# Subject controls
subject_type = widgets.ToggleButtons(
    options=[("Noun phrase", "noun"), ("Pronoun", "pronoun")],
    value="noun",
    description="Subject",
)
subject_person = widgets.Dropdown(options=[1, 2, 3], value=3, description="Person")
subject_number = widgets.Dropdown(options=[("singular", "sg"), ("plural", "pl")], value="sg", description="Number")
subject_gender = widgets.Dropdown(options=[("masc", "m"), ("fem", "f"), ("neut", "n")], value="m", description="Gender")

subject_noun = widgets.Dropdown(options=sorted(NOUNS.keys()), value="man", description="Noun")
subject_det = widgets.Dropdown(
    options=[
        ("definite", "definite"),
        ("indefinite", "indefinite"),
        ("possessive", "possessive"),
        ("demonstrative", "demonstrative"),
        ("none", "none"),
    ],
    value="definite",
    description="Determiner",
)
subject_possessor = widgets.Dropdown(
    options=[
        ("my", "my"),
        ("your", "your"),
        ("his", "his"),
        ("her", "her"),
        ("its", "its"),
        ("our", "our"),
        ("their", "their"),
    ],
    value="my",
    description="Possessor",
)
subject_adj_on = widgets.Checkbox(value=False, description="Adjective")
subject_adj = widgets.Dropdown(options=sorted(ADJECTIVES), value="big", description="Adj")

# Object controls
object_number = widgets.Dropdown(options=[("singular", "sg"), ("plural", "pl")], value="sg", description="Number")
object_gender = widgets.Dropdown(options=[("masc", "m"), ("fem", "f"), ("neut", "n")], value="n", description="Gender")
object_noun = widgets.Dropdown(options=sorted(NOUNS.keys()), value="book", description="Noun")
object_det = widgets.Dropdown(
    options=[
        ("definite", "definite"),
        ("indefinite", "indefinite"),
        ("possessive", "possessive"),
        ("demonstrative", "demonstrative"),
        ("none", "none"),
    ],
    value="indefinite",
    description="Determiner",
)
object_possessor = widgets.Dropdown(
    options=[
        ("my", "my"),
        ("your", "your"),
        ("his", "his"),
        ("her", "her"),
        ("its", "its"),
        ("our", "our"),
        ("their", "their"),
    ],
    value="my",
    description="Possessor",
)
object_adj_on = widgets.Checkbox(value=False, description="Adjective")
object_adj = widgets.Dropdown(options=sorted(ADJECTIVES), value="interesting", description="Adj")

# Verb controls
verb_choice = widgets.Dropdown(options=sorted(VERBS), value="work", description="Verb")
tense_choice = widgets.Dropdown(options=[("present", "present"), ("past", "past")], value="present", description="Tense")
sentence_type = widgets.Dropdown(
    options=[
        ("affirmative", "affirmative"),
        ("negative", "negative"),
        ("question", "question"),
        ("imperative", "imperative"),
        ("subjunctive", "subjunctive"),
    ],
    value="affirmative",
    description="Type",
)

# Output widgets
sentence_out = widgets.HTML(value="")
check_input = widgets.Text(value="", description="Your sentence")
check_button = widgets.Button(description="Check")
check_out = widgets.Textarea(value="", description="Result", layout=widgets.Layout(width="100%", height="120px"))


In [5]:
# --- Build sentence from widget state ---

def effective_person_note():
    person = subject_person.value
    note = ""
    if subject_type.value == "noun" and person != 3:
        note = "Note: noun subjects are 3rd person; using 3rd person for verb."
        person = 3
    return person, note


def subject_phrase_from_widgets():
    if subject_type.value == "pronoun":
        if subject_person.value == 3:
            key = (3, subject_number.value, subject_gender.value if subject_number.value == "sg" else "pl")
            return PRONOUNS[key]
        return PRONOUNS[(subject_person.value, subject_number.value)]

    adjective = subject_adj.value if subject_adj_on.value else None
    possessor_key = subject_possessor.value if subject_det.value == "possessive" else None
    return make_noun_phrase(
        subject_noun.value,
        number=subject_number.value,
        d_type=subject_det.value,
        adjective=adjective,
        possessor_key=possessor_key,
    )


def object_phrase_from_widgets():
    adjective = object_adj.value if object_adj_on.value else None
    possessor_key = object_possessor.value if object_det.value == "possessive" else None
    return make_noun_phrase(
        object_noun.value,
        number=object_number.value,
        d_type=object_det.value,
        adjective=adjective,
        possessor_key=possessor_key,
    )


def render_sentence():
    subj = subject_phrase_from_widgets()
    obj = object_phrase_from_widgets()
    person, note = effective_person_note()

    sentence = build_sentence(
        subj,
        verb_choice.value,
        obj,
        sentence_type=sentence_type.value,
        tense=tense_choice.value,
        person=person,
        number=subject_number.value,
    )

    if note:
        sentence_out.value = f"<b>Sentence:</b> {sentence}<br><em>{note}</em>"
    else:
        sentence_out.value = f"<b>Sentence:</b> {sentence}"

def run_check(_):
    subj = subject_phrase_from_widgets()
    obj = object_phrase_from_widgets()
    person, _ = effective_person_note()

    expected = build_sentence(
        subj,
        verb_choice.value,
        obj,
        sentence_type=sentence_type.value,
        tense=tense_choice.value,
        person=person,
        number=subject_number.value,
    )
    user = check_input.value
    if user.strip() == expected.strip():
        check_out.value = "Correct."
    else:
        check_out.value = f"Expected: {expected}\nYou wrote: {user}"

# Bind changes
all_controls = [
    subject_type, subject_person, subject_number, subject_gender,
    subject_noun, subject_det, subject_possessor, subject_adj_on, subject_adj,
    object_number, object_gender, object_noun, object_det, object_possessor, object_adj_on, object_adj,
    verb_choice, tense_choice, sentence_type,
]


def _on_change(change):
    render_sentence()


def _bind(widget, handler):
    if hasattr(widget, "_svo_handler"):
        widget.unobserve(widget._svo_handler, names="value")
    widget._svo_handler = handler
    widget.observe(handler, names="value")


for w in all_controls:
    _bind(w, _on_change)

check_button.on_click(run_check)

render_sentence()



In [6]:
# --- Layout ---

subject_box = widgets.VBox([
    subject_type,
    subject_person,
    subject_number,
    subject_gender,
    subject_det,
    subject_possessor,
    subject_adj_on,
    subject_adj,
    subject_noun,
])

verb_box = widgets.VBox([
    verb_choice,
    tense_choice,
    sentence_type,
])

object_box = widgets.VBox([
    object_number,
    object_gender,
    object_det,
    object_possessor,
    object_adj_on,
    object_adj,
    object_noun,
])

check_box = widgets.VBox([
    check_input,
    check_button,
    check_out,
])

accordion = widgets.Accordion(children=[subject_box, verb_box, object_box, check_box])
accordion.set_title(0, "Subject")
accordion.set_title(1, "Verb")
accordion.set_title(2, "Object")
accordion.set_title(3, "Check")

ui = widgets.VBox([
    sentence_out,
    accordion,
])

display(ui)
